In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from ipywidgets import FloatSlider, VBox, HBox, Label, GridspecLayout, Dropdown
from IPython.display import display, clear_output
import scipy.stats as st
from scipy.special import gamma, digamma, gammaln

In [2]:
df = pd.read_excel("Рубанов Михаил.xlsx", sheet_name='Sheet1')

In [3]:
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)

data = df.iloc[:, 1]
sample_data = np.random.choice(size=75000, a=data,)
print(f'\ndata has size={data.size}')


data has size=1000000


Итак, я загрузил данные, и теперь я **хочу изобразить полученные данные**, чтобы понять **какое оно имеет распределение**. Я хочу сделать это **динамически** (с возможностью выбора параметров распределения и самого распределения, поэтому, так как лагало при полных данных я буду брать случайную подвыборку от исходных данных, чтобы изобразить графики для первого задания)

In [4]:
np.random.seed(RANDOM_STATE)

def show_distribution(sample_data):
    # делаю графики для PDF и CDF
    fig = go.FigureWidget(
        make_subplots(
            rows=1, cols=2,
            subplot_titles=['PDF', 'CDF'],
            horizontal_spacing=0.05,
        )
    )

    # данные для теоретических и данных из лабы
    x = np.linspace(-0.05, np.max(sample_data), 10000)
    sorted_data = np.sort(sample_data)
    ecdf_y = (np.arange(1, len(sorted_data) + 1)) / len(sorted_data)

    # настраиваю первоначальный вид
    fig.add_trace(go.Scattergl(x=x, y=np.zeros_like(x), name='pdf theorethical'), row=1, col=1)
    fig.add_trace(go.Histogram(x=sorted_data, histnorm='probability density', name='histogram data'), row=1, col=1)
    fig.add_trace(go.Scattergl(x=x, y=np.zeros_like(x), name='cdf theorethical'), row=1, col=2)
    fig.add_trace(go.Scattergl(x=sorted_data, y=ecdf_y, name='empirical cdf data'), row=1, col=2)

    # делаю слайдеры для возможности изменения параметров
    slider_a = FloatSlider(value=0.2,
                            min=0.001,
                            max=5.0,
                            step=0.01,
                            continuous_update=True,
                            description='a = ',)
    slider_scale = FloatSlider(value=0.9,
                            min=0.001,
                            max=5.0,
                            step=0.005,
                            continuous_update=True,
                            description='scale = ',)
    slider_d1 = FloatSlider(value=1.0,
                            min=0.001,
                            max=5.0,
                            step=0.05,
                            continuous_update=True,
                            description='d1 = ',)
    slider_d2 = FloatSlider(value=1.0,
                            min=0.001,
                            max=5.0,
                            step=0.05,
                            continuous_update=True,
                            description='d2 = ',)
    # делаю выбор распределения
    dist_options = {
        'expon': st.expon,
        'gamma': st.gamma,
        'fisher': st.f,
        'weibull': st.dweibull
    }

    dist_dropdown = Dropdown(
        options=list(dist_options.keys()),
        value='gamma',
        description='Distribution:',
    )

    # функция для переключения динамического параметров и распределений
    def on_slider_change(*args):
        dist_name = dist_dropdown.value
        dist = dist_options[dist_name]

        if dist_name == 'expon':
            scale = slider_scale.value
            pdf_new = dist.pdf(x, scale=scale)
            cdf_new = dist.cdf(x, scale=scale)
            slider_a.layout.visibility = 'hidden'
            slider_d1.layout.visibility = 'hidden'
            slider_d2.layout.visibility = 'hidden'

        elif dist_name == 'gamma':
            a = slider_a.value
            scale = slider_scale.value
            pdf_new = dist.pdf(x, a=a, scale=scale)
            cdf_new = dist.cdf(x, a=a, scale=scale)
            slider_a.layout.visibility = 'visible'
            slider_scale.layout.visibility = 'visible'
            slider_d1.layout.visibility = 'hidden'
            slider_d2.layout.visibility = 'hidden'

        elif dist_name == 'fisher':
            d1 = slider_d1.value
            d2 = slider_d2.value
            pdf_new = dist.pdf(x, dfn=d1, dfd=d2)
            cdf_new = dist.cdf(x, dfn=d1, dfd=d2)
            slider_a.layout.visibility = 'hidden'
            slider_scale.layout.visibility = 'hidden'
            slider_d1.layout.visibility = 'visible'
            slider_d2.layout.visibility = 'visible'
        elif dist_name == 'weibull':
            scale = slider_scale.value
            pdf_new = dist.pdf(x, c=scale)
            cdf_new = dist.cdf(x, c=scale)
            slider_a.layout.visibility = 'hidden'
            slider_scale.layout.visibility = 'visible'
            slider_d1.layout.visibility = 'hidden'
            slider_d2.layout.visibility = 'hidden'



        with fig.batch_update():
            fig.data[0].y = pdf_new
            fig.data[2].y = cdf_new
            fig.layout.annotations[0].text = f'PDF + HIST - {dist_name}'
            fig.layout.annotations[1].text = f'CDF + ECDF - {dist_name}'

    # подключаю слайдеры и выпадающий список
    dist_dropdown.observe(on_slider_change, names='value')
    slider_a.observe(on_slider_change, names='value')
    slider_scale.observe(on_slider_change, names='value')
    slider_d1.observe(on_slider_change, names='value')
    slider_d2.observe(on_slider_change, names='value')


    fig.update_layout(hovermode='x unified',
                    margin=dict(l=10, r=10, b=20, t=50),
                    legend_orientation='h',
                    height=550)

    fig.update_xaxes(matches='x', showticklabels=True, row=1, col=1)
    fig.update_xaxes(matches='x', showticklabels=True, row=1, col=2)
    fig.update_yaxes(matches='y', showticklabels=True, row=1, col=1)
    fig.update_yaxes(matches='y', showticklabels=True, row=1, col=2)

    # вывожу
    display(VBox([
        HBox([dist_dropdown, slider_scale, slider_a, slider_d1, slider_d2],),
        fig
    ]))

    on_slider_change()

    fig.write_html('graphs/distribution_visualisation.html')


In [5]:
show_distribution(data)

**Вывод по графикам**: как можно увидеть, у нас наши данные имеют (скорее всего) Гамма распределение с параметрами $a\approx 0.2$ и $\alpha = 0.9$

**№2 Построить ОММ (с заданной функцией) и ОМП неизвестьных параметров
угаданного распределения.**

Найдем оценку параметров, используя метод моментов:

$$\mathbb{E}X=\lambda / \alpha\text{, } \mathbb{D}X = \lambda /\alpha^2 \rightarrow \lambda=\mathbb{D}X \cdot \alpha^2$$

$$\mathbb{E}X=\mathbb{D}X \cdot \alpha^2 / \alpha = \mathbb{D}X \cdot \alpha \rightarrow \alpha = \mathbb{E}X / \mathbb{D}, \lambda = \mathbb{E^2}X / \mathbb{D}X$$

Заменяем моменты на эмпирические и получаем:

$$\alpha = \overline{X} / S^2 \text{, } \lambda=(\overline{X})^2 / S^2$$


In [6]:
def mom_mle_show(data):
    shape_mom, _, scale_mom = st.gamma.fit(data[data > 0], floc=0, method='MM')
    shape_mle, _, scale_mle = st.gamma.fit(data[data > 0], floc=0, method='MLE')
    x = np.linspace(0, max(data), 10000)
    print(f'MM estimates for parameters: shape - {shape_mom:.4f}, scale - {scale_mom:.4f}')
    print(f'ML estimates for parameters: shape - {shape_mle:.4f}, scale - {scale_mle:.4f}')

    fig = go.FigureWidget()
    fig.add_trace(go.Histogram(x=data[data > 0], histnorm = 'probability density', name='Histogram real data'))
    fig.add_trace(go.Scatter(x=x, y=st.gamma(a=shape_mom, loc=0, scale=scale_mom).pdf(x), name='MOM gamma dist curve'))
    fig.add_trace(go.Scatter(x=x, y=st.gamma(a=shape_mle, loc=0, scale=scale_mle).pdf(x), name='MLE gamma dist curve'))

    fig.update_layout(hovermode='x unified', margin=dict(l=20, r=20, b=50, t=70), legend_orientation='h',
                    title='MM and ML estimations')

    return fig


In [7]:
fig = mom_mle_show(data)
fig.write_html('graphs/mom_ml_estimations.html')

MM estimates for parameters: shape - 0.2096, scale - 0.8935
ML estimates for parameters: shape - 0.2349, scale - 0.7973


**№3 Графически исследовать оценки на состоятельность и асимптотическую
нормальность.**

In [8]:
def log_likelihood(sample, shape, scale):
    rate = 1 / scale
    return np.sum(shape * np.log(rate) - gammaln(shape) + (shape - 1) * np.log(sample) - rate * sample, axis=1)

def mom_estimate_vectorized(sample, **kwargs):
    mean = sample.mean(axis=1)
    var = sample.var(axis=1)
    shape_est = mean ** 2 / var
    scale_est = mean / var
    return shape_est, scale_est

def mle_estimate_vectorized(sample, **kwargs):
    n_samples = sample.shape[0]
    shapes_est = []
    scales_est = []
    for n in range(n_samples):
        shape_est, _, scale_est = st.gamma.fit(sample[n, :], **kwargs)
        shapes_est.append(shape_est)
        scales_est.append(scale_est)

    return np.array(shapes_est), np.array(scales_est)

shape_mom, _, scale_mom = st.gamma.fit(data[data > 0], floc=0, method='MM')
shape_mle, _, scale_mle = st.gamma.fit(data[data > 0], floc=0, method='MLE')
n_samples = [10, 50, 100, 500, 1000, 5000, 10000, 15000, 20000]

sample_mom = st.gamma(a=shape_mom, loc=0, scale=scale_mom).rvs(size=400000000).reshape(20000, -1)
sample_mle = st.gamma(a=shape_mle, loc=0, scale=scale_mle).rvs(size=400000000).reshape(20000, -1)
print(f'Sample_mom shape - {sample_mom.shape}')
print(f'Sample_mle shape - {sample_mle.shape}')

Sample_mom shape - (20000, 20000)
Sample_mle shape - (20000, 20000)


In [9]:
def show_asymp_norm(sample, n_samples, true_shape, true_scale, estimate_method, method_name="MOM", **kwargs):   
    x = np.linspace(-3, 3, 3000)
    y = st.norm.pdf(x)

    normal_shapes = []
    normal_scales = []
    for n in n_samples:
        subsample = sample[:, :n]
        shape_est, scale_est = estimate_method(sample=subsample, **kwargs)
        normal_shape = (shape_est - true_shape) * np.sqrt(n)
        normal_scale = (scale_est - true_scale) * np.sqrt(n)

        standard_normal_shape = normal_shape / np.std(normal_shape)
        standard_normal_scale = normal_scale / np.std(normal_scale)

        normal_shapes.append(standard_normal_shape)
        normal_scales.append(standard_normal_scale)

    fig = go.Figure(
        make_subplots(
            cols=1, rows=2,
            shared_xaxes=True,
            vertical_spacing=0.08,
        )
    )

    for idx in range(len(n_samples)):
        fig.add_trace(
            go.Histogram(
                x=normal_shapes[idx],
                histnorm='probability density',
                name='Shape histogram',
                xbins={'size': 0.1},
            ),
            col=1, row=1
        )
        fig.add_trace(
            go.Scatter(
                go.Scatter(
                x=x,
                y=y,
                name='N(0, 1)',
                showlegend=False,
                )
            ),
            col=1, row=1
        )
        fig.add_trace(
            go.Histogram(
                x=normal_shapes[idx],
                histnorm='probability density',
                name='Shape histogram',
                xbins={'size': 0.1},
            ),
            col=1, row=2
        )
        fig.add_trace(
            go.Scatter(
                go.Scatter(
                x=x,
                y=y,
                name='N(0, 1)',
                showlegend=False,
                )
            ),
            col=1, row=2
        )
        

    for i in range(len(fig.data)):
        fig.data[i].visible = False

    steps = []
    for k in range(len(n_samples)):
        visible = [False] * len(fig.data)
        visible[k* 4 : k * 4 + 4] = [True] * 4

        step = dict(
            method='update',
            args = [{'visible': visible},
                    {'title': f"Asymptotic normality of {method_name} (n = {n_samples[k]})"},
                    {'label': str(n_samples[k])}
                    ]
                    )
        steps.append(step)

    sliders = [dict(
        active=0,
        currentvalue={"prefix": "n = ", "font": {"size": 14}},
        steps=steps,

    )]

    for i in range(4):
        fig.data[i].visible = True

    fig.update_layout(sliders = sliders,
                      hovermode='x unified',
                      title=f"Asymptotic normality of {method_name} (n = {n_samples[0]})",
                      margin=dict(l=30, r=30, b=100, t=70),
                      legend_orientation='h',
                      legend=dict(xanchor='right', x=0.95, y=1.1),
                      height=600)
    fig.update_xaxes(range = [-3, 3])

    return fig


In [10]:
fig = show_asymp_norm(sample_mom, n_samples, shape_mom, scale_mom, mom_estimate_vectorized, 'MOM')
fig.write_html('graphs/mom_asymp_norm.html')


In [11]:
fig = show_asymp_norm(sample_mle, n_samples,shape_mle, scale_mle, mle_estimate_vectorized, method_name='MLE', floc=0, method='mle')
fig.write_html('graphs/mle_asymp_norm.html')


In [12]:
def show_consistency(sample: np.array, n_samples: list, estimate_method, true_shape: float, true_scale: float, method_name: str, **kwargs):
    shapes_est = []
    scales_est = []
    shapes_diffs = []
    scales_diffs = []
    for n in n_samples:
        subsample = sample[0, :n]
        shape_est, _, scale_est = estimate_method(subsample, **kwargs)

        shapes_est.append(shape_est)
        scales_est.append(scale_est)

        shapes_diffs.append(np.abs(shape_est - true_shape))
        scales_diffs.append(np.abs(scale_est - true_scale))
    
    fig = go.Figure(
        make_subplots(
            cols=2, rows=2,
            subplot_titles=[
                'Shape consistensy', 'Shape estimate difference with true shape',
                'Scale consistensy', 'Scale estimate difference with true scale',
            ],
            horizontal_spacing=0.08,
            vertical_spacing=0.18,

        )
    )

    fig.add_trace(go.Scatter(
        x=n_samples, y=shapes_est,
        name=f'Shape consistensy',
        ),
        col=1,
        row=1
    )
    fig.add_hline(y=true_shape, line_dash='dash', line_color='red', col=1, row=1)    
    fig.add_trace(go.Scatter(
        x=n_samples, y=shapes_diffs,
        name=f'Shapes diffs',
        ),
        col=2,
        row=1
    )
    fig.add_trace(go.Scatter(
        x=n_samples, y=scales_est,
        name=f'Scale consistensy',
        ),
        col=1,
        row=2
    )
    fig.add_hline(y=true_scale, line_dash='dash', line_color='red', col=1, row=2)    
    fig.add_trace(go.Scatter(
        x=n_samples, y=scales_diffs,
        name=f'Scales diffs',
        ),
        col=2,
        row=2
    )

    fig.update_xaxes(title='N samples')
    fig.update_layout(title=f'Consistensy of {method_name}', margin=dict(l=30, r=30, b=50, t=70),
                      legend_orientation='h', height=700)

    return fig

n_samples = list(range(10, 200000, 5000))

In [13]:
fig = show_consistency(sample_mom.reshape(1, -1), n_samples, st.gamma.fit, shape_mom, scale_mom, 'MOM', floc=0, method='MM')
fig.write_html('graphs/mom_cons.html')


In [14]:
fig = show_consistency(sample_mle.reshape(1, -1), n_samples, st.gamma.fit, shape_mle, scale_mle, 'MLE', floc=0, method='MLE')
fig.write_html('graphs/mle_cons.html')


**Вывод**: оценки, полученные при помощи метода моментов и метода максимального правдоподобия **состоятельны и асимптотически нормальны**

**№4 Построить асимптотические доверительные интервалы уровней доверия
0.95 и 0.99.**

In [15]:
def bootstrap_CI(data, n, max_iter=2000, conf_lvl=0.95, **kwargs):
    shapes = []
    scales = []
    for _ in range(max_iter):
        subsample = np.random.choice(data, size=n, replace=True)
        shape_est, _, scale_est = st.gamma.fit(subsample, **kwargs)
        shapes.append(shape_est)
        scales.append(scale_est)

    lb = (1 - conf_lvl) / 2
    ub = conf_lvl + lb

    return np.quantile(shapes, q=[lb, ub]), np.quantile(scales, q=[lb, ub])

In [16]:
def bootstrap_sigma(data, n, max_iter=2000, **kwargs):
    shapes = []
    scales = []
    for _ in range(max_iter):
        subsample = np.random.choice(data, size=n, replace=True)
        shape_est, _, scale_est = st.gamma.fit(subsample, **kwargs)
        shapes.append(shape_est)
        scales.append(scale_est)
    
    sigma_shape = np.std(shapes) * np.sqrt(n)
    sigma_scale = np.std(scales) * np.sqrt(n)
    
    return sigma_shape, sigma_scale

def asymp_CI(data, true_shape, true_scale, n, method_name, conf_lvl, random_state=RANDOM_STATE):
    sample = st.gamma(a=true_shape, loc=0, scale=true_scale).rvs(size=n, random_state=random_state)
    shape_est, _, scale_est = st.gamma.fit(sample, floc=0, method=method_name)

    std_shape_est, std_scale_est = bootstrap_sigma(data, n)

    eps = 1 - conf_lvl

    lb_shape = shape_est - st.norm.ppf(1 - eps / 2) * std_shape_est / np.sqrt(n)
    ub_shape = shape_est + st.norm.ppf(1 - eps / 2) * std_shape_est / np.sqrt(n)

    lb_scale = scale_est - st.norm.ppf(1 - eps / 2) * std_scale_est / np.sqrt(n)
    ub_scale = scale_est + st.norm.ppf(1 - eps / 2) * std_scale_est / np.sqrt(n)

    return (lb_shape, ub_shape), (lb_scale, ub_scale)


In [17]:
def show_confidence_interval(data, n_samples, ci_method, conf_lvl, true_shape, true_scale, **kwargs):
    x = np.linspace(0.001, max(data), 6000)

    shapes_bounds = []
    scales_bounds = []
    for n in n_samples:
        shape_bounds, scale_bounds = ci_method(data=data, n=n, conf_lvl=conf_lvl, **kwargs)
        shape_lb, shape_ub = shape_bounds
        scale_lb, scale_ub = scale_bounds
        shapes_bounds.append((shape_lb, shape_ub))
        scales_bounds.append((scale_lb, scale_ub))

    fig = go.Figure(
        make_subplots(
            cols=1, rows=2,
        )
    )

    for i in range(len(n_samples)):
        fig.add_trace(go.Scatter(
            x=x,
            y=st.gamma(a=shapes_bounds[i][0], loc=0, scale=true_scale).pdf(x),
            name=f'shape_lb = {shapes_bounds[i][0]:.5f}',
            ),
            row=1, col=1
            )
        fig.add_trace(go.Scatter(
            x=x,
            y=st.gamma(a=shapes_bounds[i][1], loc=0, scale=true_scale).pdf(x),
            name=f'shape_ub = {shapes_bounds[i][1]:.5f}',
            fill='tonexty',
            ),
            row=1, col=1
            )
        
        fig.add_trace(go.Scatter(
            x=x,
            y=st.gamma(a=true_shape, loc=0, scale=scales_bounds[i][0]).pdf(x),
            name=f'scale_lb = {scales_bounds[i][0]:.5f}',
            ),
            row=2, col=1
            )
        fig.add_trace(go.Scatter(
            x=x,
            y=st.gamma(a=true_shape, loc=0, scale=scales_bounds[i][1]).pdf(x),
            name=f'scale_ub = {scales_bounds[i][1]:.5f}',
            fill='tonexty',
            ),
            row=2, col=1
            )
    

    for i in range(len(fig.data)):
        fig.data[i].visible = False

    steps = []
    for i in range(len(n_samples)):
        visible = [False] * len(fig.data)
        visible[4 * i: 4 * i + 4] = [True] * 4

        step = dict(
            method = 'update',
            args = [
                {'visible': visible},
                {'title': f'{int(conf_lvl * 100)}% Confidence Interval in 1D influence on density'},
                {'label': str(n_samples[i])}
                ],
        )
        steps.append(step)

    sliders = [dict(
        active=0,
        currentvalue={"prefix": "n = ", "font": {"size": 14}},
        steps=steps,
    )]

    for i in range(4):
        fig.data[i].visible = True

    fig.update_layout(sliders = sliders,
                      hovermode='x unified',
                      title=f'{int(conf_lvl * 100)}% Confidence Interval in 1D influence on density',
                      margin=dict(l=30, r=30, b=50, t=70),
                      legend_orientation='h',
                      height=600,
                      legend=dict(x=0.4, y=1.1))
    
    return fig

In [18]:
fig = show_confidence_interval(data[data > 0], [10, 100, 1000, 5000, 10000], bootstrap_CI, conf_lvl=0.95, true_shape=shape_mom, true_scale=scale_mom, floc=0, method='MM')
fig.write_html('graphs/mom_ci_1d.html')

In [19]:
fig = show_confidence_interval(data[data > 0], [10, 100, 1000, 5000, 10000], bootstrap_CI, conf_lvl=0.95, true_shape=shape_mle, true_scale=scale_mle, floc=0, method='mle')
fig.write_html('graphs/mle_ci_1d.html')

In [20]:
def bootstrap_conf_ellipse(data, n, max_iter=2000, conf_lvl=0.95, **kwargs):
    shapes = []
    scales = []
    
    for _ in range(max_iter):
        subsample = np.random.choice(data, size=n, replace=True)
        shape_est, _, scale_est = st.gamma.fit(subsample, **kwargs)
        shapes.append(shape_est)
        scales.append(scale_est)

    shapes = np.array(shapes)
    scales = np.array(scales)

    alpha = 1 - conf_lvl
    
    shape_low, shape_high = np.quantile(shapes, [alpha/2, 1 - alpha/2])
    scale_low, scale_high = np.quantile(scales, [alpha/2, 1 - alpha/2])

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=shapes, y=scales,
        mode='markers',
        marker=dict(size=3, opacity=0.5),
        name='Bootstrap samples'
    ))

    fig.add_vline(x=shape_low, line_dash='dash', line_color='red')
    fig.add_vline(x=shape_high, line_dash='dash', line_color='red')

    fig.add_hline(y=scale_low, line_dash='dash', line_color='blue')
    fig.add_hline(y=scale_high, line_dash='dash', line_color='blue')

    fig.add_shape(
        type="rect",
        x0=shape_low, x1=shape_high,
        y0=scale_low, y1=scale_high,
        line=dict(color="green", width=2),
        fillcolor="green",
        opacity=0.1
    )

    fig.update_xaxes(title='Shape estimates')
    fig.update_yaxes(title='Scale estimates')
    fig.update_layout(
        title=f'{int(conf_lvl*100)}% bootstrap CI in 2D',
        margin=dict(l=30, r=30, b=50, t=70),
        height=600,
    )

    return fig

In [21]:
fig = bootstrap_conf_ellipse(data[data > 0], 50000, floc=0, method='MM')
fig.write_html('graphs/ci95_2d.html')
fig.show()

In [22]:
fig = bootstrap_conf_ellipse(data[data > 0], 50000, conf_lvl=0.99, floc=0, method='MM')
fig.write_html('graphs/ci99_2d.html')
fig.show()

**№5 Проверить гипотезу о том, что выборка взята из угаданного распределе-
ния с оцененными параметрами. Использовать критерий Колмогорова.**

In [23]:
stat_mom, p_val_mom = st.ks_1samp(data[data >= 0], cdf=st.gamma(a=shape_mom, loc=0, scale=scale_mom).cdf)
stat_mle, p_val_mle = st.ks_1samp(data[data >= 0], cdf=st.gamma(a=shape_mle, loc=0, scale=scale_mle).cdf)

print(f'MOM p-value: {p_val_mom}, stat: {stat_mom}')
print(f'MLE p-value: {p_val_mle}, stat: {stat_mle}')

MOM p-value: 0.0, stat: 0.03867
MLE p-value: 0.0, stat: 0.05147250113363844


**№6 Понять, почему делать так, как в пункте 5 нельзя.**

Ответ: хоть значение `p-value` и равно нулю, то есть верна гипотеза о том, что выборка имеет Гамма распределение с параметрами, оцененными по ММ и ММП, но факт того, что гипотеза верна **не говорит в точности о том, что выборка имеет такое распределение**, оно говорит о том, что **данные не противоречат гипотезе.**

**№7 Учесть шум (вторая выборка) и проделать пункты 1-5 заново.**

In [24]:
df = pd.read_excel("Рубанов Михаил.xlsx", sheet_name='Sheet1')
noise = pd.read_excel("Рубанов Михаил.xlsx", sheet_name='Sheet2')

In [25]:
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)

noisy_data = pd.concat((df, noise), axis=0).iloc[:, 1]
sample_data = np.random.choice(size=75000, a=noisy_data,)
print(f'\ndata has size={data.size}')


data has size=1000000


In [26]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=noise.iloc[:, 1], histnorm='probability density'))
fig.update_layout(title='Noise distribution', margin=dict(l=30, r=30, b=50, t=70))
fig.write_html('graphs/noise_dist.html')
fig.show()

Как можно заметить, шум очень похож на **равномерное распределение** с параметрами 0.5 и 3.0

**№7.1 Понять, из какого распределения нам дана эта выборка.**

In [27]:
fig = go.FigureWidget(
    make_subplots(
        rows=1, cols=2,
        subplot_titles=['PDF', 'CDF'],
        horizontal_spacing=0.05,
        shared_xaxes=True,
    )
)

x = np.linspace(-0.05, max(noisy_data), 10000)
sorted_data = np.sort(noisy_data)
ecdf_y = (np.arange(1, len(sorted_data) + 1)) / len(sorted_data)

fig.add_trace(go.Histogram(x=noisy_data, histnorm='probability density', name='histogram real data'), col=1, row=1)
fig.add_trace(go.Scatter(x=x, y=st.gamma(a=shape_mle, loc=0, scale=scale_mle).pdf(x), name='gamma pdf'), col=1, row=1)

fig.add_trace(go.Scatter(x=sorted_data, y=ecdf_y, name='empirical cdf'), col=2, row=1)
fig.add_trace(go.Scatter(x=x, y=st.gamma(a=shape_mle, loc=0, scale=scale_mle).cdf(x), name='gamma cdf'), col=2, row=1)


fig.update_layout(hovermode='x unified', title='Distribution noisy data check', legend_orientation='h',margin=dict(l=30,r=30,b=50,t=70))
fig.write_html('graphs/dist_vis_noise.html')

Снова видим, что у нас Гамма распределение

**№7.2 Построить ОММ (с заданной функцией) и ОМП неизвестных параметров
угаданного распределения.**

In [28]:
fig =mom_mle_show(noisy_data[noisy_data > 0])
fig.write_html('graphs/mom_ml_estimations_noise.html')

MM estimates for parameters: shape - 0.2096, scale - 0.8935
ML estimates for parameters: shape - 0.2349, scale - 0.7973


**№7.3 Графически исследовать оценки на состоятельность и асимптотическую
нормальность.**

In [29]:
shape_mom, _, scale_mom = st.gamma.fit(noisy_data[noisy_data > 0], floc=0, method='MM')
shape_mle, _, scale_mle = st.gamma.fit(noisy_data[noisy_data > 0], floc=0, method='MLE')
n_samples = [10, 50, 100, 500, 1000, 5000, 10000, 15000, 20000]

sample_mom = st.gamma(a=shape_mom, loc=0, scale=scale_mom).rvs(size=400000000).reshape(20000, -1)
sample_mle = st.gamma(a=shape_mle, loc=0, scale=scale_mle).rvs(size=400000000).reshape(20000, -1)
print(f'Sample_mom shape - {sample_mom.shape}')
print(f'Sample_mle shape - {sample_mle.shape}')

Sample_mom shape - (20000, 20000)
Sample_mle shape - (20000, 20000)


In [30]:
fig = show_asymp_norm(sample_mom, n_samples, shape_mom, scale_mom, estimate_method=mom_estimate_vectorized, method_name='MOM')
fig.write_html('graphs/mom_asymp_norm_noise.html')

In [31]:
fig = show_asymp_norm(sample_mle, n_samples,shape_mle, scale_mle, mle_estimate_vectorized, method_name='MLE', floc=0, method='mle')
fig.write_html('graphs/mle_asymp_norm_noise.html')

In [32]:
n_samples = list(range(10, 50010, 2500))
fig = show_consistency(sample_mom.reshape(1, -1), n_samples, st.gamma.fit, shape_mom, scale_mom, 'MOM', floc=0, method='MM')
fig.write_html('graphs/mom_cons_noise.html')

In [33]:
fig = show_consistency(sample_mle.reshape(1, -1), n_samples, st.gamma.fit, shape_mle, scale_mle, 'MLE', floc=0, method='MLE')
fig.write_html('graphs/mle_cons_noise.html')

**Вывод**: оценки на зашумленных данных **асимптотически нормальны и состоятельны**

**№7.4 Построить асимптотические доверительные интервалы уровней доверия 0.95 и 0.99**

In [34]:
fig = show_confidence_interval(noisy_data[noisy_data > 0], [10, 100, 1000, 5000, 10000], bootstrap_CI, conf_lvl=0.95, true_shape=shape_mom, true_scale=scale_mom, floc=0, method='MM')
fig.write_html('graphs/mom_ci_1d_noise.html')

In [35]:
fig = show_confidence_interval(noisy_data[noisy_data > 0], [10, 100, 1000, 5000, 10000], bootstrap_CI, conf_lvl=0.95, true_shape=shape_mle, true_scale=scale_mle, floc=0, method='mle')
fig.write_html('graphs/mle_ci_1d_noise.html')

In [36]:
fig = bootstrap_conf_ellipse(noisy_data[noisy_data > 0], 50000, floc=0, method='MM')
fig.write_html('graphs/ci95_2d_noise.html')

In [37]:
fig = bootstrap_conf_ellipse(noisy_data[noisy_data > 0], 50000, conf_lvl=0.99, floc=0, method='MM')
fig.write_html('graphs/ci99_2d_noise.html')

**№7.5 Проверить гипотезу о том, что выборка взята из угаданного распределе-
ния с оцененными параметрами. Использовать критерий Колмогорова.**

In [38]:
stat_mom, p_val_mom = st.ks_1samp(noisy_data[noisy_data >= 0], cdf=st.gamma(a=shape_mom, loc=0, scale=scale_mom).cdf)
stat_mle, p_val_mle = st.ks_1samp(noisy_data[noisy_data >= 0], cdf=st.gamma(a=shape_mle, loc=0, scale=scale_mle).cdf)

print(f'MOM p-value: {p_val_mom}, stat: {stat_mom}')
print(f'MLE p-value: {p_val_mle}, stat: {stat_mle}')

MOM p-value: 0.0, stat: 0.03867
MLE p-value: 0.0, stat: 0.05147250113363844


Видим, что тест не изменился в сравнении с тестом без шума

**Вывод насчет шума**: Добавленный шум не повлиял на оценки параметров через ОМП и ОММ. На тесты он также не повлиял 